# Zerobus Ingest Benchmark Driver

Run all ingest modes (gRPC sync/async, HTTP/1.1 sync/async, HTTP/2 sync/async)
across a configurable number of iterations using `zbhelper.ingest_v2`.

Select which modes to run via the **widgets in Step 3**, then run all cells top to bottom.
Results are appended to `benchmark_results.jsonl` and displayed as a flattened table.

### Step 1: Install dependencies

In [ ]:
%pip install --quiet databricks-sdk[notebook] databricks-zerobus-ingest-sdk aiohttp requests httpx[http2]

In [ ]:
# DO NOT add %autoreload here — it resets the module-level OAuth token cache
# (_token_cache in ingest_v2.py) on every cell execution, forcing a token
# re-fetch every run.  If you edit zbhelper, restart the kernel manually
# (Kernel → Restart) and re-run from this cell.

import sys
from pathlib import Path

_nb = next((str(p) for p in [Path.cwd(), Path.cwd() / "notebooks"] if (p / "zbhelper").is_dir()), None)
if _nb and _nb not in sys.path:
    sys.path.insert(0, _nb)

import zbhelper.ingest_v2 as zbv2
import zbhelper.setup as zbsetup
print("zbhelper loaded")

### Step 2: Connection & table configuration

`zbhelper.setup` auto-discovers the workspace URL, region, and ZeroBus endpoint, then
ensures the service principal and OAuth secret exist. Set `SP_NAME` below — everything
else is derived automatically.

In [ ]:
# ── User-configurable ────────────────────────────────────────────────────
SP_NAME = "lfcdemo_zerobus"   # service principal display name (drives secret scope)

# ── Step 2.a: auto-discover workspace + ZeroBus endpoint ─────────────────
_ws = zbsetup.discover_workspace(dbutils)
DATABRICKS_WORKSPACE_URL = _ws["workspace_url"]
DATABRICKS_WORKSPACE_ID  = _ws["workspace_id"]
ZEROBUS_INGEST_URL       = _ws["zerobus_ingest_url"]
SERVER_ENDPOINT          = _ws["server_endpoint"]

# ── Step 2.b + 2.c: ensure SP exists; mint/validate OAuth secret ──────────
_sp = zbsetup.ensure_service_principal(_ws, dbutils, sp_name=SP_NAME)
CLIENT_ID     = _sp["client_id"]
CLIENT_SECRET = _sp["client_secret"]

### Step 3: Benchmark widgets

Select which modes, concurrency levels, and iteration count to run.
Defaults run **all 6 modes** × concurrency 1 × 10 iterations.

In [ ]:
# ── Canonical option sets — only place to add/remove values ──────────────
_APIS_ALL          = ["grpc", "http/1.1", "http/2"]
_SYNC_ASYNCS_ALL   = ["sync", "async"]
_CONCURRENCIES_ALL = ["1", "2", "4", "8", "16", "32"]

# On Databricks: "all" selects every option; single-value default only (ipywidgets limitation)
dbutils.widgets.multiselect("api",         "all",  ["all"] + _APIS_ALL)
dbutils.widgets.multiselect("sync_async",  "all",  ["all"] + _SYNC_ASYNCS_ALL)
dbutils.widgets.multiselect("concurrency", "all",    ["all"] + _CONCURRENCIES_ALL)
dbutils.widgets.dropdown(   "iters",       "10",   ["1","5","10","20","50"])
dbutils.widgets.dropdown(   "n",           "1000", ["100","500","1000","5000"])

In [ ]:
def _sel(name, all_vals):
    """Return all_vals if widget is unset or "all", else the selected subset."""
    v = dbutils.widgets.get(name).split(",")
    return all_vals if "all" in v else v

_APIS          = _sel("api",         _APIS_ALL)
_SYNC_ASYNCS   = _sel("sync_async",  _SYNC_ASYNCS_ALL)
_CONCURRENCIES = _sel("concurrency", _CONCURRENCIES_ALL)
_ITERS         = int(dbutils.widgets.get("iters"))
_N             = int(dbutils.widgets.get("n"))

_MODES = []
for _a in _APIS:
    for _s in _SYNC_ASYNCS:
        _m = zbv2.MODE_MAP.get((_a, _s))
        if _m:
            _MODES.append(_m)
        else:
            print(f"Warning: unsupported combination api={_a!r} sync_async={_s!r} — skipped")

print(f"Modes to run : {_MODES}")
print(f"Concurrencies: {_CONCURRENCIES}")
print(f"Iters × N    : {_ITERS} × {_N}")
print(f"Total runs   : {len(_MODES) * len(_CONCURRENCIES) * _ITERS}")

### Step 4: Create tables

One table per mode (`airquality_<mode>`). Tables are created if they don't exist and
the service principal is granted the required UC privileges.

In [ ]:
# Create one UC table per selected mode; cache catalog/schema for the cfg dict.
_tables: dict[str, dict] = {}
for _m in _MODES:
    _tbl = zbsetup.ensure_table(spark, _ws, CLIENT_ID, table=f"airquality_{_m}")
    _tables[_m] = _tbl
    print(f"  {_m:15s} → {_tbl['table_name']}")

CATALOG = _tables[_MODES[0]]["catalog"]
SCHEMA  = _tables[_MODES[0]]["schema"]
print(f"\nCATALOG={CATALOG!r}  SCHEMA={SCHEMA!r}")

### Step 5: Benchmark loop

**Once before the loop:**
- Network ping — TCP SYN/ACK to host:443 (`ping_ms`, no TLS or HTTP)

**Once per mode (first time that mode is seen):**
- HTTP ping — warm GET on the open session, no insert (`http_ping_ms`, HTTP modes only)
- OAuth token fetch — cached for all subsequent iterations of the same mode

**For each (mode × concurrency × iteration):**
- Sync modes (`grpc_sync`, `http_sync`, `http2_sync`) with `concurrency > 1` are **skipped** — concurrency is meaningless for blocking calls.
1. Opens a stream / HTTP session (`setup_zerobus`; reuses cached OAuth token)
2. **4a** — launches all `min(10, N)` single-row inserts via `asyncio.gather`; semaphore caps in-flight calls at `concurrency`
3. **4b** — launches all 10 batch runs via `asyncio.gather`; same semaphore caps in-flight calls at `concurrency`
4. Closes the session and polls UC visibility
5. Appends the result to `benchmark_results.jsonl`

In [ ]:
import asyncio as _asyncio
import datetime
import time

from pathlib import Path
from statistics import mean, median

_RESULTS_FILE = Path(".") / "benchmark_results.jsonl"
_BATCH_RUNS   = 10   # number of batch runs per iteration (matches zerobus_grpc_http.ipynb)

# ── one-time baseline measurements ───────────────────────────────────────────
_ping_s = zbv2.ping_endpoint(ZEROBUS_INGEST_URL)
print(f"Network ping: {_ping_s * 1000:.1f} ms")

_mode_http_ping_s: dict[str, float | None] = {}  # measured once per mode

for _iter in range(1, _ITERS + 1):
    for _mode in _MODES:
        _is_async = _mode.endswith("_async")
        _concurrencies = [int(c) for c in _CONCURRENCIES]
        for _concurrency in _concurrencies:
            if _concurrency > 1 and not _is_async:
                print(f"skip: {_mode} is sync — concurrency={_concurrency} has no effect")
                continue
            print(f"\n{'='*64}")
            print(f"iter={_iter}/{_ITERS}  mode={_mode}  concurrency={_concurrency}")
            print(f"{'='*64}")

            _tbl        = _tables[_mode]
            _table_name = _tbl["table_name"]

            _cfg = {
                "server_endpoint":    SERVER_ENDPOINT,
                "workspace_url":      DATABRICKS_WORKSPACE_URL,
                "workspace_id":       DATABRICKS_WORKSPACE_ID,
                "zerobus_ingest_url": ZEROBUS_INGEST_URL,
                "client_id":          CLIENT_ID,
                "client_secret":      CLIENT_SECRET,
                "catalog":            _tbl["catalog"],
                "schema":             _tbl["schema"],
                "table_name":         _table_name,
            }

            # ── build records ─────────────────────────────────────────────
            _singles_n = min(10, _N)
            _batch_n   = _N - _singles_n
            _records_4a    = zbv2.build_records(_singles_n)
            _batch_records = zbv2.build_records(_batch_n, offset=_singles_n)

            # ── baseline ──────────────────────────────────────────────────
            _baseline = zbv2.fetch_row_baseline(spark, _table_name)
            _row_before = _baseline["count"]

            # ── setup ─────────────────────────────────────────────────────
            _zb_client = await zbv2.setup_zerobus(_mode, _cfg)
            _connect_s = _zb_client["_connect_s"]
            _oauth_s   = _zb_client["_oauth_s"]

            # ── HTTP ping (once per mode for entire run, HTTP modes only) ──
            if _mode not in _mode_http_ping_s:
                _http_ping_s = await zbv2.http_ping(_zb_client, ZEROBUS_INGEST_URL)
                _mode_http_ping_s[_mode] = _http_ping_s
                if _http_ping_s is not None:
                    print(f"HTTP ping ({_mode}): {_http_ping_s * 1000:.1f} ms")
            else:
                _http_ping_s = _mode_http_ping_s[_mode]

            # ── 4a: single-row inserts ────────────────────────────────────
            _row_send_s: list[float] = []
            _row_wait_s: list[float] = []
            _row_ack_s:  list[float] = []

            _sem = _asyncio.Semaphore(_concurrency)

            async def _insert_one(rec):
                async with _sem:
                    return await zbv2.call_zerobus_insert(_zb_client, [rec])

            _t_4a0 = time.perf_counter()
            _res_4a = await _asyncio.gather(*[_insert_one(r) for r in _records_4a])
            _singles_wall_s = time.perf_counter() - _t_4a0
            for _s, _w in _res_4a:
                _row_send_s.append(_s)
                _row_wait_s.append(_w)
                _row_ack_s.append(_s + _w)

            # ── 4b: batch inserts ─────────────────────────────────────────
            _batch_send_s: list[float] = []
            _batch_wait_s: list[float] = []
            _batch_run_s:  list[float] = []
            _disconnect_s  = 0.0
            _t_after_close = time.perf_counter()

            async def _insert_batch():
                async with _sem:
                    return await zbv2.call_zerobus_insert(_zb_client, _batch_records)

            try:
                _res_4b = await _asyncio.gather(*[_insert_batch() for _ in range(_BATCH_RUNS)])
                for _s, _w in _res_4b:
                    _batch_send_s.append(_s)
                    _batch_wait_s.append(_w)
                    _batch_run_s.append(_s + _w)
            finally:
                _tc = time.perf_counter()
                await _zb_client["close"]()
                _t_after_close = time.perf_counter()
                _disconnect_s  = _t_after_close - _tc

            _ingest_4a4b_s = _singles_wall_s + sum(_batch_run_s)
            _total_rows    = _singles_n + _batch_n * len(_batch_run_s)

            # ── 4c: visibility ────────────────────────────────────────────
            _vis = zbv2.poll_visibility(
                spark, _table_name, _row_before + _total_rows,
                _t_after_close, _t_4a0,
            )
            _visibility_s                  = _vis["visibility_s"]
            _visibility_from_first_send_s  = _vis["visibility_from_first_send_s"]

            # ── 4d: print metrics ─────────────────────────────────────────
            print(f"\nIngested {_total_rows} rows → {_table_name}  [mode={_mode}]")
            print(f"  visibility: {_visibility_from_first_send_s*1000:.1f} ms (from first send → COUNT(*) target)")
            print(f"  visibility: {_visibility_s*1000:.1f} ms (from end of ingest → COUNT(*) target)")
            if _oauth_s is not None:
                print(f"  oauth:      {_oauth_s*1000:.1f} ms")
            print(f"  connect:    {_connect_s*1000:.1f} ms")
            print(f"  disconnect: {_disconnect_s*1000:.1f} ms")
            print(f"  ingest wall (4a+4b): {_ingest_4a4b_s*1000:.1f} ms")
            if _row_ack_s:
                print(f"  4a ({_singles_n} singles) wall {_singles_wall_s*1000:.1f} ms"
                      f"  send+wait: min={min(_row_ack_s)*1000:.1f}  "
                      f"median={median(_row_ack_s)*1000:.1f}  max={max(_row_ack_s)*1000:.1f} ms")
            if _batch_run_s:
                print(f"  4b ({_batch_n} rows × {len(_batch_run_s)} runs)"
                      f"  wall min={min(_batch_run_s)*1000:.1f}  "
                      f"median={median(_batch_run_s)*1000:.1f}  max={max(_batch_run_s)*1000:.1f} ms")

            # ── 4e: append result ─────────────────────────────────────────
            _result = {
                "run_at":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
                "mode":             _mode,
                "mode_label":       zbv2.MODE_LABEL.get(_mode, _mode),
                "table":            _table_name,
                "zerobus_endpoint": ZEROBUS_INGEST_URL,
                "n":                _N,
                "concurrency":      _concurrency,
                "oauth_ms":         zbv2.ms(_oauth_s)      if _oauth_s      is not None else None,
                "ping_ms":          zbv2.ms(_ping_s),
                "http_ping_ms":  zbv2.ms(_http_ping_s) if _http_ping_s is not None else None,
                "connect_ms":       zbv2.ms(_connect_s)    if _connect_s    is not None else None,
                "disconnect_ms":    zbv2.ms(_disconnect_s) if _disconnect_s is not None else None,
                "ingest_wall_ms":   zbv2.ms(_ingest_4a4b_s),
                "visibility_from_first_send_ms": zbv2.ms(_visibility_from_first_send_s),
                "visibility_from_end_ms":        zbv2.ms(_visibility_s),
                "4a": {
                    "rows":         1,
                    "runs":         _singles_n,
                    "wall_ms":      zbv2.ms(_singles_wall_s),
                    "send_ms":      zbv2.stats_ms(_row_send_s),
                    "wait_ms":      zbv2.stats_ms(_row_wait_s),
                    "send_wait_ms": zbv2.stats_ms(_row_ack_s),
                } if _row_ack_s else None,
                "4b": {
                    "rows":         _batch_n,
                    "runs":         len(_batch_run_s),
                    "wall_ms":      zbv2.ms(sum(_batch_run_s)),
                    "send_wait_ms": zbv2.stats_ms(_batch_run_s),
                    "send_ms":      zbv2.stats_ms(_batch_send_s),
                    "wait_ms":      zbv2.stats_ms(_batch_wait_s),
                } if _batch_run_s else None,
            }
            zbv2.append_jsonl(_result, _RESULTS_FILE)
            print(f"  → appended to {_RESULTS_FILE}")

### Step 6: Results

Display all results from `benchmark_results.jsonl` as a flat table.
When the same `(mode, scenario)` appears more than once, numeric columns are
aggregated to the **median** and a `run_count` column shows how many runs were merged.

In [ ]:
zbv2.display_results(_RESULTS_FILE)

### Step 7: Export to CSV

In [ ]:
import os
_csv_path = Path(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)) if "__vsc_ipynb_file__" in vars() else ".") / "benchmark_results.csv"
_df = zbv2.flatten_jsonl_to_df(_RESULTS_FILE)
zbv2.write_csv_from_df(_df, _csv_path)